# LOCUS — ARC-AGI Training Notebook

This notebook runs the LOCUS companion agent against ARC-AGI games using the Anthropic API.
LOCUS reads `companion_arcprize.md` as its system prompt and picks actions based on its game-mechanic knowledge graph.

**Before running:** add `ANTHROPIC_API_KEY` and `ARC_API_KEY` under  
*notebook → Settings → Add-ons → Secrets* then toggle each secret on for this notebook.

In [ ]:
# Install dependencies
!pip install anthropic arc-agi -q

In [ ]:
# Inject Kaggle secrets into environment
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
os.environ["ARC_API_KEY"]       = secrets.get_secret("ARC_API_KEY")

In [ ]:
# Download kaggle_agent.py from the companion_arc repo
!wget -q https://raw.githubusercontent.com/antfriend/companion_arc/main/kaggle_agent.py

In [ ]:
# Setup — fetches companion_arcprize.md and initialises the Anthropic client
from kaggle_agent import setup, run_training_attempt

client, companion = setup()

In [ ]:
# Run one training attempt in practice mode
# Change game_id to the target task (e.g. "ls20")
# Set max_steps high enough to cover the level sequence
result = run_training_attempt(
    game_id="ls20",
    client=client,
    companion_text=companion,
    max_steps=300,
    competition_mode=False,   # flip to True for leaderboard submission
    verbose=True,
)

print("\n--- Result ---")
print(f"steps:            {result['steps']}")
print(f"levels_completed: {result['levels_completed']}")
print(f"final_state:      {result['final_state']}")
print(f"scorecard:\n{result['scorecard']}")

## Logging a result back to LOCUS

After a run, log the outcome so LOCUS can update its knowledge graph in `companion_arcprize.md`.
Copy the log text below and paste it as an `@LOCUS LOG` command in Claude Code:

```
@LOCUS LOG level N complete — [ai_actions] actions, baseline [human_baseline], score [(baseline/ai)²] — [brief mechanic observation]
```

## Switching to competition mode

When ready for a leaderboard submission, change the `run_training_attempt` call above to `competition_mode=True`.  
Competition constraints: API-only, no local resets, single environment, score is final on `get_scorecard()`.